# LeetCode 65 — Valid Number (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def isNumber(self, s: str) -> bool:

```

Bài này được xếp **Hard** không phải vì thuật toán khó, mà vì **rất dễ sai chi tiết**: dấu `+`/`-` đứng ở đâu, dấu chấm `.` được phép khi nào, sau chữ `e` phải có gì... Đây là bài kinh điển để học cách **biến một định nghĩa bằng lời thành code chính xác**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc định nghĩa "số hợp lệ" của đề như một **ngữ pháp** (grammar) gồm các mảnh nhỏ: integer, decimal, exponent.
2. Vì sao cách "lười" `float(s)` + `try/except` **trả lời sai** dù chạy qua hết ví dụ của đề.
3. Cách 2 — tách chuỗi theo `e`/`E` rồi viết `is_integer`, `is_decimal`: dễ hiểu nhất.
4. Cách 3 — **một lượt quét** với 3 cờ `seen_digit`, `seen_dot`, `seen_exp`: cách chính, trace kỹ.
5. Cách 4 — **máy trạng thái hữu hạn DFA** (Deterministic Finite Automaton) học từ con số 0, có bảng chuyển trạng thái bằng `dict`.
6. Cách 5 — **regex** với `re.fullmatch`, giải thích từng ký hiệu.
7. Kiểm thử lớn: danh sách valid/invalid của đề + kiểm tra **vét cạn** mọi chuỗi ngắn để đối chiếu chéo tất cả các cách.

## 1. Đề bài (bản hiện hành trên LeetCode)

Cho một chuỗi `s`, trả về `True` nếu `s` là **số hợp lệ** (valid number), ngược lại trả về `False`.

Định nghĩa hình thức của đề (giữ nguyên tiếng Anh cho chính xác):

> A **valid number** is defined using one of the following definitions:
> 1. An **integer number** followed by an optional **exponent**.
> 2. A **decimal number** followed by an optional **exponent**.
>
> An **integer number** is defined with an optional sign `'-'` or `'+'` followed by **digits**.
>
> A **decimal number** is defined with an optional sign `'-'` or `'+'` followed by one of the following definitions:
> 1. **Digits** followed by a dot `'.'`.
> 2. **Digits** followed by a dot `'.'` followed by **digits**.
> 3. A dot `'.'` followed by **digits**.
>
> An **exponent** is defined with an exponent notation `'e'` or `'E'` followed by an **integer number**.
>
> The **digits** is defined as one or more digits.

Đề cho sẵn hai danh sách:

- **Hợp lệ:** `"2"`, `"0089"`, `"-0.1"`, `"+3.14"`, `"4."`, `"-.9"`, `"2e10"`, `"-90E3"`, `"3e+7"`, `"+6e-1"`, `"53.5e93"`, `"-123.456e789"`
- **Không hợp lệ:** `"abc"`, `"1a"`, `"1e"`, `"e3"`, `"99e2.5"`, `"--6"`, `"-+3"`, `"95a54e53"`

| Ví dụ | Input `s` | Output |
|---|---|---|
| 1 | `"0"` | `true` |
| 2 | `"e"` | `false` |
| 3 | `"."` | `false` |

**Constraints:**

- `1 <= s.length <= 20`
- `s` chỉ gồm chữ cái tiếng Anh (cả hoa lẫn thường), chữ số `0-9`, dấu cộng `'+'`, dấu trừ `'-'`, hoặc dấu chấm `'.'`.

> **Lưu ý về phiên bản đề.** Bản hiện hành ở trên **không có khoảng trắng**: constraints không cho ký tự `' '` xuất hiện, nên ta không cần (và không nên) `strip()`. Bản đề rất cũ của bài này từng cho phép khoảng trắng ở đầu/cuối (kiểu `" 0.1 "` → `true`); notebook này bám bản hiện hành, còn bản có khoảng trắng được để thành một **bài tập biến thể** ở cuối.
>
> Một chi tiết quan trọng khác: constraints **cho phép chữ cái**, nên những input như `"inf"`, `"nan"`, `"abc"` hoàn toàn có thể xuất hiện trong test. Ta sẽ thấy vì sao điều này "giết" cách dùng `float()`.

In [1]:
VALID_FROM_PROBLEM = ["2", "0089", "-0.1", "+3.14", "4.", "-.9", "2e10", "-90E3",
                      "3e+7", "+6e-1", "53.5e93", "-123.456e789", "0"]
INVALID_FROM_PROBLEM = ["abc", "1a", "1e", "e3", "99e2.5", "--6", "-+3", "95a54e53",
                        "e", "."]

print("Số chuỗi hợp lệ (đề + ví dụ)    :", len(VALID_FROM_PROBLEM))
print("Số chuỗi không hợp lệ (đề + ví dụ):", len(INVALID_FROM_PROBLEM))
print("Chuỗi dài nhất:", max(VALID_FROM_PROBLEM + INVALID_FROM_PROBLEM, key=len))

Số chuỗi hợp lệ (đề + ví dụ)    : 13
Số chuỗi không hợp lệ (đề + ví dụ): 10
Chuỗi dài nhất: -123.456e789


## 2. Đọc đề như một "ngữ pháp"

Định nghĩa của đề thực ra là một **ngữ pháp** (grammar): một số hợp lệ được ghép từ các "mảnh" nhỏ hơn. Viết gọn lại (dấu `[...]` nghĩa là *có thể có hoặc không*):

```text
number   = (integer | decimal) [exponent]
integer  = [sign] digits
decimal  = [sign] ( digits "."  |  digits "." digits  |  "." digits )
exponent = ("e" | "E") integer
sign     = "+" | "-"
digits   = một hoặc nhiều ký tự từ 0 đến 9
```

Ba điều cần khắc ghi ngay từ đầu:

1. **`digits` là "một hoặc nhiều"** chữ số, không được rỗng. Vì vậy `"."`, `"+"`, `"1e"` đều sai.
2. **Phần sau `e` phải là `integer`**: được có dấu, **không** được có dấu chấm. Vì vậy `"99e2.5"` sai.
3. **Dấu chỉ đứng ở 2 chỗ**: đầu chuỗi, hoặc ngay sau `e`/`E`. Vì vậy `"--6"`, `"-+3"`, `"1+2"` sai.

Chú ý: đề **không cấm số 0 ở đầu** (`"0089"` hợp lệ) và cho phép `"4."` (có chấm mà không có phần lẻ) lẫn `"-.9"` (không có phần nguyên).

### Tính tay: tách vài chuỗi thành các mảnh

| `s` | Tách mảnh | Kết luận |
|---|---|---|
| `"-123.456e789"` | sign `-` · digits `123` · `.` · digits `456` · `e` · integer `789` | decimal + exponent → **True** |
| `"4."` | digits `4` · `.` | decimal dạng "digits ." → **True** |
| `"-.9"` | sign `-` · `.` · digits `9` | decimal dạng ". digits" → **True** |
| `"3e+7"` | digits `3` · `e` · integer `+7` | integer + exponent → **True** |
| `"99e2.5"` | `99` · `e` · `2.5` | `2.5` không phải integer → **False** |
| `"1e"` | `1` · `e` · (rỗng) | sau `e` thiếu digits → **False** |
| `"e3"` | (rỗng) · `e` · `3` | trước `e` không có số → **False** |
| `"."` | `.` | decimal cần ít nhất một bên có digits → **False** |
| `"--6"` | `-` · `-6` | chỉ được 1 dấu → **False** |

Mọi cách giải phía dưới chỉ là **những cách khác nhau để kiểm tra đúng ngữ pháp này**.

## 3. Kiến thức Python nền tảng

### 3.1. Duyệt từng ký tự và phân loại ký tự

- `for c in s:` lấy lần lượt từng ký tự.
- `for i, c in enumerate(s):` lấy cả **vị trí** `i` lẫn ký tự `c` (cần khi muốn nhìn ký tự đứng trước `s[i - 1]`).
- `c in "+-"` kiểm tra `c` có phải một trong các ký tự `+`, `-` không.
- `'0' <= c <= '9'`: so sánh ký tự theo **mã số Unicode** (xem bằng `ord`). Các chữ số `'0'..'9'` có mã liên tiếp `48..57`, nên phép so sánh kép này đúng nghĩa "c là chữ số ASCII".

Vì sao không dùng `c.isdigit()`? Nó trả `True` cho cả những "chữ số" lạ như `'²'` hay chữ số Ả Rập `'١'`. Trong constraints của bài thì những ký tự đó không xuất hiện, nhưng thói quen viết `'0' <= c <= '9'` giúp ta **đúng định nghĩa** mà không phụ thuộc may rủi.

In [2]:
s = "-1.5e3"
for i, c in enumerate(s):
    print(i, repr(c), "| chữ số?", '0' <= c <= '9', "| dấu?", c in "+-", "| e?", c in "eE")

print("ord('0') =", ord('0'), " ord('9') =", ord('9'))
print("'²'.isdigit() =", '²'.isdigit(), "  nhưng '0' <= '²' <= '9' =", '0' <= '²' <= '9')

0 '-' | chữ số? False | dấu? True | e? False
1 '1' | chữ số? True | dấu? False | e? False
2 '.' | chữ số? False | dấu? False | e? False
3 '5' | chữ số? True | dấu? False | e? False
4 'e' | chữ số? False | dấu? False | e? True
5 '3' | chữ số? True | dấu? False | e? False
ord('0') = 48  ord('9') = 57
'²'.isdigit() = True   nhưng '0' <= '²' <= '9' = False


### 3.2. Cắt chuỗi, `split`, `count`, `all(...)`

- `t[1:]` bỏ ký tự đầu (dùng để bỏ dấu `+`/`-`).
- `t.split("e")` cắt chuỗi tại mọi chữ `e`, trả về **list** các mảnh. Nếu `e` nằm ở đầu/cuối, sẽ xuất hiện mảnh rỗng `''`.
- `t.count(".")` đếm số dấu chấm.
- `all(điều_kiện for c in t)` trả `True` nếu **mọi** ký tự thỏa điều kiện. Cẩn thận: `all(...)` trên chuỗi rỗng trả `True` (không có phản ví dụ nào), nên phải tự kiểm tra `len(t) > 0`.

In [3]:
print("'+42'[1:]        ->", repr('+42'[1:]))
print("'2e10'.split('e') ->", '2e10'.split('e'))
print("'1e'.split('e')   ->", '1e'.split('e'))
print("'e3'.split('e')   ->", 'e3'.split('e'))
print("'1e2e3'.split('e')->", '1e2e3'.split('e'))
print("'1.2.3'.count('.')->", '1.2.3'.count('.'))
print("all(...) trên chuỗi rỗng ->", all('0' <= c <= '9' for c in ''))

'+42'[1:]        -> '42'
'2e10'.split('e') -> ['2', '10']
'1e'.split('e')   -> ['1', '']
'e3'.split('e')   -> ['', '3']
'1e2e3'.split('e')-> ['1', '2', '3']
'1.2.3'.count('.')-> 2
all(...) trên chuỗi rỗng -> True


### 3.3. `try` / `except`: bắt lỗi thay vì để chương trình dừng

```python
try:
    # đoạn code có thể gây lỗi
except ValueError:
    # chạy khi đoạn trên ném ra ValueError
```

`float("abc")` ném `ValueError`. Bọc nó trong `try/except` ta biến "có lỗi / không lỗi" thành `False` / `True`. Đây chính là nền của cách "lười" ngay sau đây.

In [4]:
for t in ["3.14", "abc"]:
    try:
        value = float(t)
        print(repr(t), "-> float được:", value)
    except ValueError as err:
        print(repr(t), "-> ValueError:", err)

'3.14' -> float được: 3.14
'abc' -> ValueError: could not convert string to float: 'abc'


## 4. Cách 1 — "Lười": nhờ `float()` kiểm tra hộ

Ý tưởng người mới hay nghĩ ra đầu tiên: Python đã biết đọc số thực rồi, vậy cứ thử `float(s)`; không lỗi thì là số.

Cách này có vẻ rất hợp lý, và quả thật nó **qua hết** các ví dụ đề cho. Hãy chạy thử.

In [5]:
def is_number_float(s: str) -> bool:
    try:
        float(s)
        return True
    except ValueError:
        return False

print("Valid của đề   :", all(is_number_float(t) for t in VALID_FROM_PROBLEM))
print("Invalid của đề :", not any(is_number_float(t) for t in INVALID_FROM_PROBLEM))

Valid của đề   : True
Invalid của đề : True


### Vì sao cách này SAI — bài học "đừng tin công cụ có định nghĩa khác đề"

`float()` dùng **ngữ pháp số thực của Python**, không phải ngữ pháp của đề. Hai ngữ pháp *gần giống nhau* nhưng Python chấp nhận thêm nhiều thứ:

| Input | `float()` | Đề bài | Vì sao Python chấp nhận | Có nằm trong constraints? |
|---|---|---|---|---|
| `"inf"`, `"Infinity"`, `"-INF"` | nhận | **False** | Python hiểu là vô cực | **Có** (chỉ gồm chữ cái và dấu) |
| `"nan"`, `"NaN"` | nhận | **False** | "Not a Number" | **Có** |
| `"1_000"` | nhận | False | dấu `_` phân cách (từ Python 3.6) | Không (`_` không được phép) |
| `" 1 "` | nhận | False | `float` tự bỏ khoảng trắng | Không (bản đề hiện hành) |
| `"١٢٣"` | nhận | False | chữ số Unicode | Không |

Dòng `inf`/`nan` là chí mạng: các chuỗi đó **hợp lệ theo constraints** (chỉ toàn chữ cái), nên LeetCode hoàn toàn có thể cho test `"inf"` và chờ `false`. Cách "lười" sẽ **Wrong Answer**.

Bài học tổng quát: khi đề định nghĩa một khái niệm (số hợp lệ, email hợp lệ, IP hợp lệ...), hãy **cài đặt đúng định nghĩa đó**. Đừng mượn một hàm thư viện có định nghĩa "na ná".

In [6]:
traps = ["inf", "Infinity", "-INF", "nan", "NaN", "1_000", " 1 ", "١٢٣"]
for t in traps:
    print(f"{t!r:12} -> float() nói {is_number_float(t)}   (đề bài: False)")

'inf'        -> float() nói True   (đề bài: False)
'Infinity'   -> float() nói True   (đề bài: False)
'-INF'       -> float() nói True   (đề bài: False)
'nan'        -> float() nói True   (đề bài: False)
'NaN'        -> float() nói True   (đề bài: False)
'1_000'      -> float() nói True   (đề bài: False)
' 1 '        -> float() nói True   (đề bài: False)
'١٢٣'        -> float() nói True   (đề bài: False)


Có thể "vá" bằng cách cấm trước `inf`, `nan`, `_`, khoảng trắng... Nhưng vá kiểu danh sách đen (blacklist) luôn mong manh: ta phải **biết hết** những gì Python lỡ chấp nhận. Cách đáng tin là tự kiểm tra theo ngữ pháp. Ta vẫn giữ `is_number_float` lại cho một việc hữu ích ở phần test: trên bảng chữ cái chỉ gồm chữ số, `+`, `-`, `.`, `e`, `E` (không có chữ cái nào khác) thì ngữ pháp Python **trùng** với ngữ pháp đề, nên nó dùng được làm "trọng tài" đối chiếu.

## 5. Cách 2 — Tách theo `e`/`E`, viết đúng từng định nghĩa

Đây là cách **dễ hiểu nhất** vì nó "dịch" thẳng ngữ pháp sang code.

Nhìn lại: `number = (integer | decimal) [exponent]` và `exponent = e integer`. Vậy:

1. Đổi `E` thành `e`, rồi `split("e")`.
2. Nếu có **nhiều hơn 2 mảnh** → có từ 2 chữ `e` trở lên → `False`.
3. Mảnh đầu (gọi là `base`) phải là `integer` **hoặc** `decimal`.
4. Nếu có mảnh thứ hai (phần mũ) thì nó phải là `integer`.

Ta cần 3 hàm nhỏ, mỗi hàm đúng một định nghĩa:

- `is_digits(t)`: `t` khác rỗng và toàn chữ số.
- `is_integer(t)`: bỏ một dấu ở đầu (nếu có), phần còn lại là `digits`.
- `is_decimal(t)`: bỏ một dấu ở đầu, phần còn lại có **đúng một** dấu chấm; tách thành `left`/`right`, mỗi bên rỗng hoặc toàn chữ số, và **không được cả hai cùng rỗng**.

Bảng "3 dạng decimal" của đề khớp với điều kiện cuối thế nào?

| `left` | `right` | Dạng trong đề | Hợp lệ? |
|---|---|---|---|
| `"4"` | `""` | digits `.` | có |
| `"3"` | `"14"` | digits `.` digits | có |
| `""` | `"9"` | `.` digits | có |
| `""` | `""` | (chỉ có dấu chấm) | **không** |

In [7]:
def is_digits(t: str) -> bool:
    return len(t) > 0 and all('0' <= c <= '9' for c in t)

def strip_sign(t: str) -> str:
    # bỏ tối đa MỘT dấu ở đầu
    if t and t[0] in "+-":
        return t[1:]
    return t

def is_integer(t: str) -> bool:
    return is_digits(strip_sign(t))

def is_decimal(t: str) -> bool:
    t = strip_sign(t)
    if t.count(".") != 1:
        return False
    left, right = t.split(".")
    if left == "" and right == "":
        return False
    return (left == "" or is_digits(left)) and (right == "" or is_digits(right))

for t in ["+42", "--6", "", "4.", "-.9", "3.14", ".", "1.2.3", "+.", "1a.5"]:
    print(f"{t!r:8} is_integer={is_integer(t)!s:5}  is_decimal={is_decimal(t)}")

'+42'    is_integer=True   is_decimal=False
'--6'    is_integer=False  is_decimal=False
''       is_integer=False  is_decimal=False
'4.'     is_integer=False  is_decimal=True
'-.9'    is_integer=False  is_decimal=True
'3.14'   is_integer=False  is_decimal=True
'.'      is_integer=False  is_decimal=False
'1.2.3'  is_integer=False  is_decimal=False
'+.'     is_integer=False  is_decimal=False
'1a.5'   is_integer=False  is_decimal=False


In [8]:
def is_number_split(s: str) -> bool:
    parts = s.replace("E", "e").split("e")
    if len(parts) > 2:
        return False
    base = parts[0]
    if not (is_integer(base) or is_decimal(base)):
        return False
    if len(parts) == 2 and not is_integer(parts[1]):
        return False
    return True

print("Valid của đề   :", all(is_number_split(t) for t in VALID_FROM_PROBLEM))
print("Invalid của đề :", not any(is_number_split(t) for t in INVALID_FROM_PROBLEM))
print("Bẫy inf/nan    :", [is_number_split(t) for t in ["inf", "nan", "Infinity"]])

Valid của đề   : True
Invalid của đề : True
Bẫy inf/nan    : [False, False, False]


### Trace cách 2

| `s` | `parts` | `base` hợp lệ? | phần mũ là integer? | Kết quả |
|---|---|---|---|---|
| `"-123.456e789"` | `['-123.456', '789']` | decimal ✔ | `'789'` ✔ | True |
| `"99e2.5"` | `['99', '2.5']` | integer ✔ | `'2.5'` ✘ | False |
| `".e1"` | `['.', '1']` | ✘ (`'.'` không phải decimal) | (không cần xét) | False |
| `"1e2e3"` | `['1', '2', '3']` | (không cần xét) | (không cần xét) | False vì 3 mảnh |

In [9]:
def trace_split(s: str) -> bool:
    parts = s.replace("E", "e").split("e")
    print(f"s = {s!r} -> parts = {parts}")
    if len(parts) > 2:
        print("   nhiều hơn 1 chữ e -> False")
        return False
    base = parts[0]
    print(f"   base = {base!r}: is_integer={is_integer(base)}, is_decimal={is_decimal(base)}")
    if not (is_integer(base) or is_decimal(base)):
        print("   base không hợp lệ -> False")
        return False
    if len(parts) == 2:
        print(f"   phần mũ = {parts[1]!r}: is_integer={is_integer(parts[1])}")
        if not is_integer(parts[1]):
            print("   phần mũ không phải integer -> False")
            return False
    print("   -> True")
    return True

for t in ["-123.456e789", "99e2.5", ".e1", "1e2e3"]:
    trace_split(t)

s = '-123.456e789' -> parts = ['-123.456', '789']
   base = '-123.456': is_integer=False, is_decimal=True
   phần mũ = '789': is_integer=True
   -> True
s = '99e2.5' -> parts = ['99', '2.5']
   base = '99': is_integer=True, is_decimal=False
   phần mũ = '2.5': is_integer=False
   phần mũ không phải integer -> False
s = '.e1' -> parts = ['.', '1']
   base = '.': is_integer=False, is_decimal=False
   base không hợp lệ -> False
s = '1e2e3' -> parts = ['1', '2', '3']
   nhiều hơn 1 chữ e -> False


### Độ phức tạp và điểm vướng của cách 2

Gọi `n = len(s)` (theo đề `n <= 20`).

- **Thời gian:** `O(n)`: `replace`, `split`, `count`, `all` mỗi cái đi qua chuỗi một lần (hằng số lần).
- **Bộ nhớ:** `O(n)`: tạo nhiều chuỗi con (`parts`, `left`, `right`, chuỗi sau khi bỏ dấu).

Cách này **đúng và rất rõ ràng**, hoàn toàn có thể nộp. Điểm vướng:

- Đi qua chuỗi **nhiều lượt** và tạo nhiều chuỗi tạm.
- Logic bị rải ra nhiều hàm; khi đề đổi luật (ví dụ thêm khoảng trắng, thêm `_`) phải sửa nhiều chỗ.

Câu hỏi để cải tiến: *có thể quyết định chỉ trong **một lượt** đi từ trái sang phải, chỉ nhớ vài biến `True/False` không?*

## 6. Cách 3 (cách chính) — Một lượt quét với 3 cờ

### Mấu chốt: khi đi từ trái sang phải, ta chỉ cần nhớ 3 sự thật

| Cờ (flag) | Ý nghĩa |
|---|---|
| `seen_digit` | Trong **phần đang đọc** (phần trước `e`, hoặc phần mũ sau `e`) đã gặp ít nhất một chữ số chưa? |
| `seen_dot` | Đã gặp dấu chấm `.` chưa? |
| `seen_exp` | Đã gặp `e`/`E` chưa? |

*Cờ* (flag) chỉ là một biến `bool` dùng để "đánh dấu đã xảy ra chuyện gì".

### Luật cho từng loại ký tự

| Gặp ký tự | Hợp lệ khi | Cập nhật |
|---|---|---|
| chữ số `0-9` | luôn được | `seen_digit = True` |
| dấu `+`/`-` | ở vị trí `i == 0` **hoặc** ngay sau `e`/`E` (`s[i-1] in "eE"`) | (không đổi cờ) |
| dấu chấm `.` | **chưa** có dấu chấm **và chưa** có `e` | `seen_dot = True` |
| `e`/`E` | **chưa** có `e` **và** đã có chữ số ở phần trước | `seen_exp = True`, **`seen_digit = False`** |
| ký tự khác | không bao giờ | `return False` ngay |

Hết chuỗi: trả về `seen_digit`, vì phần cuối cùng (phần chính hoặc phần mũ) phải có ít nhất một chữ số.

Mỗi luật ứng với một câu trong đề:

- Dấu chấm bị cấm sau `e` vì **exponent là integer** (không có chấm).
- `e` cần `seen_digit` vì trước nó phải là integer/decimal (mà cả hai đều có digits).
- Dấu chỉ ở đầu hoặc sau `e` vì **chỉ integer/decimal mới có dấu, và chúng chỉ bắt đầu ở hai chỗ đó**.

### Vì sao phải `seen_digit = False` khi gặp `e`?

Vì sau `e` là một **integer mới**, và integer này cũng cần "một hoặc nhiều chữ số". Nếu không đặt lại cờ, chữ số ở phần trước sẽ "đếm hộ" cho phần mũ:

```text
"1e"   : gặp '1' -> seen_digit = True; gặp 'e' -> nếu KHÔNG reset, hết chuỗi trả True  (SAI)
         có reset -> seen_digit = False, hết chuỗi trả False                         (ĐÚNG)
```

Đây là chi tiết quan trọng nhất của cách 3. Ở phần "Lỗi thường gặp" ta sẽ chạy bản quên reset để thấy nó sai.

In [10]:
def is_number_flags(s: str) -> bool:
    seen_digit = False
    seen_dot = False
    seen_exp = False

    for i, c in enumerate(s):
        if '0' <= c <= '9':
            seen_digit = True
        elif c in "+-":
            if i > 0 and s[i - 1] not in "eE":
                return False
        elif c == ".":
            if seen_dot or seen_exp:
                return False
            seen_dot = True
        elif c in "eE":
            if seen_exp or not seen_digit:
                return False
            seen_exp = True
            seen_digit = False       # phần mũ phải có chữ số của riêng nó
        else:
            return False             # chữ cái khác, ký tự lạ

    return seen_digit

In [11]:
print("Valid của đề   :", all(is_number_flags(t) for t in VALID_FROM_PROBLEM))
print("Invalid của đề :", not any(is_number_flags(t) for t in INVALID_FROM_PROBLEM))
for t in ["0", "e", ".", "4.", "-.9", "3.e+0", "+.e1", "inf"]:
    print(f"{t!r:8} -> {is_number_flags(t)}")

Valid của đề   : True
Invalid của đề : True
'0'      -> True
'e'      -> False
'.'      -> False
'4.'     -> True
'-.9'    -> True
'3.e+0'  -> True
'+.e1'   -> False
'inf'    -> False


### Trace bằng tay: `s = "-.9E+7"` (hợp lệ, đi qua gần đủ các nhánh)

Chuỗi này có: dấu ở đầu, dấu chấm khi **chưa** có chữ số, `E` hoa, dấu ngay sau `E`.

| i | c | Loại | Kiểm tra | `seen_digit` | `seen_dot` | `seen_exp` |
|---|---|---|---|---|---|---|
| 0 | `-` | dấu | `i == 0` → được | False | False | False |
| 1 | `.` | chấm | chưa chấm, chưa e → được | False | **True** | False |
| 2 | `9` | số | luôn được | **True** | True | False |
| 3 | `E` | mũ | chưa e, đã có số → được; **reset** `seen_digit` | **False** | True | **True** |
| 4 | `+` | dấu | `s[3] = 'E'` → được | False | True | True |
| 5 | `7` | số | luôn được | **True** | True | True |
| hết | | | trả `seen_digit` | **True** | | |

Và hai chuỗi không hợp lệ, mỗi chuỗi bị loại ở một chỗ khác nhau:

- `"4e1.5"`: tới `.` ở `i = 3` thì `seen_exp = True` → `False` ngay (phần mũ không có chấm).
- `"1e"`: không bị loại giữa chừng, nhưng hết chuỗi `seen_digit = False` (đã reset ở `e`) → `False`.

In [12]:
def char_kind(c: str) -> str:
    if '0' <= c <= '9': return "số"
    if c in "+-": return "dấu"
    if c == ".": return "chấm"
    if c in "eE": return "mũ"
    return "khác"

def trace_flags(s: str) -> bool:
    seen_digit = seen_dot = seen_exp = False
    print(f"s = {s!r}")
    for i, c in enumerate(s):
        kind = char_kind(c)
        ok = True
        if kind == "số":
            seen_digit = True
        elif kind == "dấu":
            ok = not (i > 0 and s[i - 1] not in "eE")
        elif kind == "chấm":
            ok = not (seen_dot or seen_exp)
            if ok: seen_dot = True
        elif kind == "mũ":
            ok = not seen_exp and seen_digit
            if ok: seen_exp, seen_digit = True, False
        else:
            ok = False
        print(f"  i={i} c={c!r} {kind:4} ok={ok!s:5} digit={seen_digit!s:5} dot={seen_dot!s:5} exp={seen_exp}")
        if not ok:
            print("  -> False (loại giữa chừng)")
            return False
    print(f"  hết chuỗi -> trả seen_digit = {seen_digit}")
    return seen_digit

trace_flags("-.9E+7")
trace_flags("4e1.5")
_ = trace_flags("1e")

s = '-.9E+7'
  i=0 c='-' dấu  ok=True  digit=False dot=False exp=False
  i=1 c='.' chấm ok=True  digit=False dot=True  exp=False
  i=2 c='9' số   ok=True  digit=True  dot=True  exp=False
  i=3 c='E' mũ   ok=True  digit=False dot=True  exp=True
  i=4 c='+' dấu  ok=True  digit=False dot=True  exp=True
  i=5 c='7' số   ok=True  digit=True  dot=True  exp=True
  hết chuỗi -> trả seen_digit = True
s = '4e1.5'
  i=0 c='4' số   ok=True  digit=True  dot=False exp=False
  i=1 c='e' mũ   ok=True  digit=False dot=False exp=True
  i=2 c='1' số   ok=True  digit=True  dot=False exp=True
  i=3 c='.' chấm ok=False digit=True  dot=False exp=True
  -> False (loại giữa chừng)
s = '1e'
  i=0 c='1' số   ok=True  digit=True  dot=False exp=False
  i=1 c='e' mũ   ok=True  digit=False dot=False exp=True
  hết chuỗi -> trả seen_digit = False


### Code nộp LeetCode — Cách 3 (một lượt quét với cờ)

In [13]:
class Solution:
    def isNumber(self, s: str) -> bool:
        seen_digit = seen_dot = seen_exp = False
        for i, c in enumerate(s):
            if '0' <= c <= '9':
                seen_digit = True
            elif c in "+-":
                if i > 0 and s[i - 1] not in "eE":
                    return False
            elif c == ".":
                if seen_dot or seen_exp:
                    return False
                seen_dot = True
            elif c in "eE":
                if seen_exp or not seen_digit:
                    return False
                seen_exp = True
                seen_digit = False
            else:
                return False
        return seen_digit


sol = Solution()
for t in VALID_FROM_PROBLEM:
    assert sol.isNumber(t) is True, t
for t in INVALID_FROM_PROBLEM:
    assert sol.isNumber(t) is False, t
# edge case dễ sai
for t, expected in [("+", False), ("-.", False), ("+.e1", False), ("1e+", False),
                    ("3.e+0", True), (".0", True), ("0.", True), ("1+2", False),
                    ("inf", False), ("nan", False), ("1E5", True), ("e", False)]:
    assert sol.isNumber(t) == expected, (t, expected)
print("class Solution: tất cả test PASS")

class Solution: tất cả test PASS


### Vì sao cách 3 đúng? (invariant bằng lời)

**Invariant** (bất biến) là một điều luôn đúng sau mỗi vòng lặp. Ở đây: sau khi đọc xong `s[0..i]` mà chưa bị `return False`:

1. `seen_exp` đúng bằng "`s[0..i]` có chứa `e`/`E`" (và chỉ chứa **đúng một** cái, vì cái thứ hai bị loại).
2. `seen_dot` đúng bằng "có dấu chấm", và dấu chấm đó (nếu có) nằm **trước** `e`; chỉ có **một** dấu chấm.
3. `seen_digit` đúng bằng "**phần hiện tại** đã có ít nhất một chữ số", với phần hiện tại là phần mũ nếu `seen_exp`, ngược lại là phần chính.
4. Mọi dấu `+`/`-` đã gặp đều nằm ở đầu chuỗi hoặc ngay sau `e`.

Từ đó, khi hết chuỗi:

- **Không trả nhầm `True`:** nếu trả `True` thì phần chính có số, có tối đa một chấm, dấu chỉ ở đầu → nó là integer (không chấm) hoặc decimal (có chấm, ít nhất một bên có số). Nếu có `e` thì phần chính có số (kiểm tra lúc gặp `e`), phần mũ không có chấm, dấu chỉ ở đầu phần mũ và có số → là integer. Đúng ngữ pháp.
- **Không bỏ sót:** một chuỗi đúng ngữ pháp thì mọi ký tự đều qua kiểm tra (dấu chỉ ở 2 vị trí cho phép, một chấm trước `e`, `e` xuất hiện sau khi phần chính đã có số) và phần cuối có số → trả `True`.

Một tinh tế: dấu sau `e` chỉ được kiểm tra bằng `s[i-1] in "eE"`, nên `"1e+-2"` bị loại ở dấu `-` (vì `s[i-1] = '+'`). Còn chuỗi `"+-"` ở đầu bị loại vì `-` ở `i = 1` mà `s[0] = '+'`.

### Độ phức tạp

Với `n = len(s)`:

- **Thời gian:** `O(n)`: mỗi ký tự xử lý một lần với vài phép so sánh.
- **Bộ nhớ:** `O(1)`: chỉ 3 biến `bool` và chỉ số `i`, không tạo chuỗi con.

## 7. Cách 4 — Máy trạng thái hữu hạn (DFA), học từ căn bản

### 7.1. DFA là gì?

**DFA** (Deterministic Finite Automaton, *máy trạng thái hữu hạn đơn định*) là một "cỗ máy" rất đơn giản:

- Có một số hữu hạn **trạng thái** (state). Mỗi trạng thái tóm tắt "đến giờ mình đã đọc được cái gì".
- Có một **trạng thái bắt đầu** (start state).
- Có **bảng chuyển trạng thái** (transition table): đang ở trạng thái `X`, đọc ký tự loại `t` thì chuyển sang trạng thái `Y`. *Đơn định* nghĩa là với mỗi cặp `(X, t)` có **tối đa một** `Y`.
- Nếu không có chuyển nào cho `(X, t)` → máy "kẹt" → chuỗi **bị loại**.
- Một số trạng thái được đánh dấu **chấp nhận** (accepting state). Đọc hết chuỗi mà đang đứng ở trạng thái chấp nhận → chuỗi **hợp lệ**.

Hình dung như chơi cờ cá ngựa: mỗi ký tự là một lần tung xúc xắc quyết định ô tiếp theo; về đích (ô chấp nhận) đúng lúc hết lượt thì thắng.

### 7.2. Ví dụ mini: nhận dạng `integer` = `[sign] digits`

| Trạng thái | gặp `digit` | gặp `sign` | Chấp nhận? |
|---|---|---|---|
| `start` | → `digits` | → `sign` | không |
| `sign` | → `digits` | — | không |
| `digits` | → `digits` | — | **có** |

Dấu `—` là "không có đường đi": kẹt, loại ngay. Trong Python, bảng này là một **`dict` lồng `dict`**: `TABLE[state][kind]` cho ra trạng thái tiếp theo.

In [14]:
INT_TABLE = {
    "start":  {"digit": "digits", "sign": "sign"},
    "sign":   {"digit": "digits"},
    "digits": {"digit": "digits"},
}
INT_ACCEPT = {"digits"}

def kind_of(c: str) -> str:
    if '0' <= c <= '9': return "digit"
    if c in "+-": return "sign"
    if c == ".": return "dot"
    if c in "eE": return "exp"
    return "other"

def is_integer_dfa(t: str) -> bool:
    state = "start"
    for c in t:
        k = kind_of(c)
        if k not in INT_TABLE[state]:     # không có đường đi -> kẹt
            return False
        state = INT_TABLE[state][k]
    return state in INT_ACCEPT

for t in ["+42", "7", "-", "4-2", "--1", ""]:
    print(f"{t!r:6} -> {is_integer_dfa(t)}")

'+42'  -> True
'7'    -> True
'-'    -> False
'4-2'  -> False
'--1'  -> False
''     -> False


### 7.3. Thiết kế DFA cho cả bài

Ta cần các trạng thái ghi nhớ đủ thông tin để quyết định bước tiếp theo:

| Trạng thái | Nghĩa là "đã đọc được..." | Chấp nhận? | Ví dụ chuỗi dừng ở đây |
|---|---|---|---|
| `start` | chưa đọc gì | không | `""` |
| `sign` | một dấu ở đầu | không | `"-"` |
| `int` | (dấu) + digits | **có** | `"-12"` |
| `dot_after_int` | digits rồi `.` | **có** | `"4."` |
| `dot_no_int` | `.` mà **chưa** có digit nào trước | không | `"-."` |
| `frac` | có `.` và digits sau chấm | **có** | `"3.14"`, `".9"` |
| `exp` | vừa đọc `e`/`E` | không | `"2e"` |
| `exp_sign` | `e` rồi dấu | không | `"2e+"` |
| `exp_int` | `e` (dấu) digits | **có** | `"2e10"` |

Vì sao phải tách `dot_after_int` và `dot_no_int`? Cả hai đều "vừa gặp dấu chấm", nhưng `"4."` hợp lệ còn `"."` thì không, và `"4.e1"` hợp lệ còn `".e1"` thì không. Hai tình huống cư xử khác nhau → phải là **hai trạng thái khác nhau**. Đây là cách nghĩ cốt lõi khi thiết kế DFA.

### 7.4. Sơ đồ trạng thái dạng bảng (ma trận chuyển)

| Từ \ gặp | `digit` | `sign` | `dot` | `exp` |
|---|---|---|---|---|
| `start` | `int` | `sign` | `dot_no_int` | — |
| `sign` | `int` | — | `dot_no_int` | — |
| **`int`** | `int` | — | `dot_after_int` | `exp` |
| **`dot_after_int`** | `frac` | — | — | `exp` |
| `dot_no_int` | `frac` | — | — | — |
| **`frac`** | `frac` | — | — | `exp` |
| `exp` | `exp_int` | `exp_sign` | — | — |
| `exp_sign` | `exp_int` | — | — | — |
| **`exp_int`** | `exp_int` | — | — | — |

(In đậm = trạng thái chấp nhận. Cột `other` toàn là `—` nên không vẽ.)

Đường đi "đầy đủ nhất" của một số hợp lệ:

```text
start --sign--> sign --digit--> [int] --dot--> [dot_after_int] --digit--> [frac]
      --e--> exp --sign--> exp_sign --digit--> [exp_int]
```

Các nhánh tắt: `start --dot--> dot_no_int --digit--> [frac]` (dạng `".9"`), `[int] --e--> exp` (dạng `"2e10"`), `[dot_after_int] --e--> exp` (dạng `"4.e1"`), và mỗi trạng thái "digits" (`int`, `frac`, `exp_int`) đều có vòng tự lặp khi gặp thêm chữ số.

In [15]:
TRANSITIONS = {
    "start":         {"digit": "int",     "sign": "sign",     "dot": "dot_no_int"},
    "sign":          {"digit": "int",                         "dot": "dot_no_int"},
    "int":           {"digit": "int",                         "dot": "dot_after_int", "exp": "exp"},
    "dot_after_int": {"digit": "frac",                                                "exp": "exp"},
    "dot_no_int":    {"digit": "frac"},
    "frac":          {"digit": "frac",                                                "exp": "exp"},
    "exp":           {"digit": "exp_int", "sign": "exp_sign"},
    "exp_sign":      {"digit": "exp_int"},
    "exp_int":       {"digit": "exp_int"},
}
ACCEPTING = {"int", "dot_after_int", "frac", "exp_int"}

def is_number_dfa(s: str) -> bool:
    state = "start"
    for c in s:
        k = kind_of(c)
        if k not in TRANSITIONS[state]:
            return False
        state = TRANSITIONS[state][k]
    return state in ACCEPTING

print("Số trạng thái:", len(TRANSITIONS), "| số cạnh chuyển:", sum(len(v) for v in TRANSITIONS.values()))
print("Valid của đề   :", all(is_number_dfa(t) for t in VALID_FROM_PROBLEM))
print("Invalid của đề :", not any(is_number_dfa(t) for t in INVALID_FROM_PROBLEM))

Số trạng thái: 9 | số cạnh chuyển: 17
Valid của đề   : True
Invalid của đề : True


### Trace DFA: `"+3.e-2"` (hợp lệ, dùng nhánh `dot_after_int -> exp`) và `"+.e1"` (bị kẹt)

| Bước | Ký tự | Loại | Trạng thái trước | Trạng thái sau |
|---|---|---|---|---|
| 1 | `+` | sign | `start` | `sign` |
| 2 | `3` | digit | `sign` | `int` |
| 3 | `.` | dot | `int` | `dot_after_int` |
| 4 | `e` | exp | `dot_after_int` | `exp` |
| 5 | `-` | sign | `exp` | `exp_sign` |
| 6 | `2` | digit | `exp_sign` | `exp_int` |
| hết | | | `exp_int` ∈ ACCEPTING | **True** |

Với `"+.e1"`: `start -sign-> sign -dot-> dot_no_int`, rồi gặp `e`: hàng `dot_no_int` không có cột `exp` → **kẹt → False**. Đúng: `"+."` chưa phải integer/decimal nên không được gắn exponent.

In [16]:
def trace_dfa(s: str) -> bool:
    state = "start"
    print(f"s = {s!r}")
    for step, c in enumerate(s, start=1):
        k = kind_of(c)
        if k not in TRANSITIONS[state]:
            print(f"  bước {step}: {c!r} ({k}) từ {state}: KHÔNG có đường đi -> False")
            return False
        nxt = TRANSITIONS[state][k]
        print(f"  bước {step}: {c!r} ({k:5}) {state:>13} -> {nxt}")
        state = nxt
    ok = state in ACCEPTING
    print(f"  hết chuỗi ở {state}: {'chấp nhận' if ok else 'không chấp nhận'} -> {ok}")
    return ok

trace_dfa("+3.e-2")
trace_dfa("+.e1")
_ = trace_dfa("2e")

s = '+3.e-2'
  bước 1: '+' (sign )         start -> sign
  bước 2: '3' (digit)          sign -> int
  bước 3: '.' (dot  )           int -> dot_after_int
  bước 4: 'e' (exp  ) dot_after_int -> exp
  bước 5: '-' (sign )           exp -> exp_sign
  bước 6: '2' (digit)      exp_sign -> exp_int
  hết chuỗi ở exp_int: chấp nhận -> True
s = '+.e1'
  bước 1: '+' (sign )         start -> sign
  bước 2: '.' (dot  )          sign -> dot_no_int
  bước 3: 'e' (exp) từ dot_no_int: KHÔNG có đường đi -> False
s = '2e'
  bước 1: '2' (digit)         start -> int
  bước 2: 'e' (exp  )           int -> exp
  hết chuỗi ở exp: không chấp nhận -> False


### So sánh DFA với cách 3 (cờ)

Thật ra **cách 3 chính là một DFA được nén**: bộ ba `(seen_digit, seen_dot, seen_exp)` cộng với "ký tự trước có phải `e` không" đóng vai trò trạng thái. DFA tường minh có lợi:

- **Luật nằm trong dữ liệu (bảng), không nằm trong `if/elif`.** Muốn đổi luật, ta sửa bảng.
- Dễ **chứng minh**: chỉ cần kiểm tra từng ô của bảng khớp ngữ pháp.
- Dùng được cho việc khác ngoài kiểm tra, ví dụ **đếm** số chuỗi hợp lệ (bài tập 6).

Độ phức tạp: thời gian `O(n)` (mỗi ký tự một lần tra `dict` trung bình `O(1)`); bộ nhớ `O(1)` vì bảng có kích thước cố định (9 trạng thái, 5 loại ký tự), không phụ thuộc `n`.

## 8. Cách 5 — Regex với `re.fullmatch`

**Regex** (regular expression, *biểu thức chính quy*) là một ngôn ngữ mini để mô tả mẫu chuỗi. Điều thú vị: mọi regex cổ điển đều tương đương một DFA, nên bài này viết được bằng **một dòng** regex.

Các ký hiệu cần dùng:

| Ký hiệu | Nghĩa | Ví dụ |
|---|---|---|
| `[0-9]` | một ký tự trong khoảng `0..9` | khớp `"7"` |
| `[+-]`, `[eE]` | một ký tự trong tập liệt kê | khớp `"+"` hoặc `"-"` |
| `X?` | `X` xuất hiện 0 hoặc 1 lần | `[+-]?` = dấu tùy chọn |
| `X+` | 1 lần trở lên | `[0-9]+` = **digits** |
| `X*` | 0 lần trở lên | `[0-9]*` |
| `\.` | dấu chấm thật (vì `.` trần nghĩa là "ký tự bất kỳ") | |
| `A\|B` | `A` hoặc `B` | |
| `(?:...)` | nhóm lại (không lưu kết quả) để áp `?`, `\|` lên cả nhóm | |

Ba hàm hay nhầm:

- `re.fullmatch(p, s)`: **cả chuỗi** phải khớp. Đây là thứ ta cần.
- `re.match(p, s)`: chỉ cần **phần đầu** khớp (`"1a"` vẫn khớp `[0-9]+` ở phần `"1"`).
- `re.search(p, s)`: khớp ở **bất kỳ đâu**.

In [17]:
import re

print(re.fullmatch(r"[0-9]+", "123"))            # khớp cả chuỗi
print(re.fullmatch(r"[0-9]+", "12a"))            # None: còn 'a' không khớp
print(re.match(r"[0-9]+", "12a"))                # match vẫn khớp phần '12'!
print([bool(re.fullmatch(r"a?b", t)) for t in ["b", "ab", "aab"]])
print([bool(re.fullmatch(r"(?:ab|c)+", t)) for t in ["abab", "cab", "ac"]])

<re.Match object; span=(0, 3), match='123'>
None
<re.Match object; span=(0, 2), match='12'>
[True, True, False]
[True, True, False]


### Ghép pattern theo đúng ngữ pháp

```text
[+-]?                         dấu tùy chọn (chung cho integer và decimal)
(?:
    [0-9]+\.?[0-9]*           digits, rồi có thể có chấm, rồi có thể có thêm digits
                              -> phủ "4", "4.", "3.14"
  | \.[0-9]+                  chấm rồi digits -> phủ ".9"
)
(?:[eE][+-]?[0-9]+)?          exponent tùy chọn: e/E, dấu tùy chọn, digits
```

Vì sao nhánh đầu viết `[0-9]+\.?[0-9]*` lại đủ? Nó cho 3 khả năng: không chấm (`"4"`, đúng là integer), có chấm không phần lẻ (`"4."`), có chấm có phần lẻ (`"3.14"`). Nhánh hai `\.[0-9]+` bắt buộc có digits sau chấm, nên `"."` không lọt.

Dùng `[0-9]` thay vì `\d`: trong Python 3, `\d` khớp **mọi chữ số Unicode** (ví dụ `'١'`), rộng hơn định nghĩa của đề, giống cái bẫy của `float()`.

In [18]:
NUMBER_RE = re.compile(r"[+-]?(?:[0-9]+\.?[0-9]*|\.[0-9]+)(?:[eE][+-]?[0-9]+)?")

def is_number_regex(s: str) -> bool:
    return NUMBER_RE.fullmatch(s) is not None

print("Valid của đề   :", all(is_number_regex(t) for t in VALID_FROM_PROBLEM))
print("Invalid của đề :", not any(is_number_regex(t) for t in INVALID_FROM_PROBLEM))
weird = "١٢٣"
print(r"\d khớp chữ số Ả Rập?", bool(re.fullmatch(r"\d+", weird)), "| [0-9] khớp?", bool(re.fullmatch(r"[0-9]+", weird)))

Valid của đề   : True
Invalid của đề : True
\d khớp chữ số Ả Rập? True | [0-9] khớp? False


Độ phức tạp: với pattern này, `fullmatch` chạy **thực tế `O(n)`**. Lưu ý engine `re` của Python dùng *backtracking* (thử nhánh, sai thì lùi lại), nên regex viết ẩu (lồng `+` trong `+`) có thể chậm theo cấp số mũ; pattern của ta chỉ có các nhánh rời nhau nên an toàn. Bộ nhớ `O(1)` ngoài đối tượng pattern đã biên dịch (`re.compile` một lần, dùng lại nhiều lần).

Regex ngắn nhất, nhưng trong phỏng vấn người ta thường hỏi tiếp *"không dùng regex thì sao?"*, nên cách 3 hoặc 4 vẫn là thứ phải viết được.

## 9. Kiểm thử lớn và đối chiếu chéo các cách

Kế hoạch test:

1. **Danh sách của đề** (12 valid + 8 invalid) và 3 ví dụ, cộng thêm các edge case dễ sai.
2. **Vét cạn** (exhaustive): mọi chuỗi độ dài 1..5 trên bảng chữ cái nhỏ `"1.eE+-a"` (19 607 chuỗi). Bốn cách đúng phải cho cùng một kết quả, và trùng với "trọng tài" `float()` (trên bảng chữ cái này `float()` không gặp được `inf`/`nan`/`_`/khoảng trắng nên an toàn).
3. **Ngẫu nhiên**: 3000 chuỗi dài tới 20 (đúng giới hạn đề), `random.seed` cố định để chạy lại ra y hệt.

In [19]:
CORRECT_METHODS = {
    "split": is_number_split,
    "flags": is_number_flags,
    "dfa":   is_number_dfa,
    "regex": is_number_regex,
}

EDGE_CASES = [
    ("+", False), ("-", False), ("+.", False), (".e1", False), ("1e+", False),
    ("1e-", False), ("+e", False), ("1.2.3", False), ("1e2e3", False), ("1+2", False),
    ("+-1", False), ("E5", False), ("6e6.5", False), ("1e5.", False), ("..1", False),
    ("inf", False), ("nan", False), ("Infinity", False),
    ("1E5", True), (".0", True), ("0.", True), ("00", True), ("3.e+0", True),
    ("46.e3", True), ("-0", True), ("+.8", True), ("1e+0", True), ("9" * 20, True),
]

ALL_CASES = ([(t, True) for t in VALID_FROM_PROBLEM]
             + [(t, False) for t in INVALID_FROM_PROBLEM] + EDGE_CASES)

for name, f in CORRECT_METHODS.items():
    for t, expected in ALL_CASES:
        assert f(t) == expected, (name, t, expected)
    print(f"{name:6} PASS {len(ALL_CASES)} ca (đề + edge case)")

float_wrong = [t for t, expected in ALL_CASES if is_number_float(t) != expected]
print("float() trả sai ở:", float_wrong)

split  PASS 51 ca (đề + edge case)
flags  PASS 51 ca (đề + edge case)
dfa    PASS 51 ca (đề + edge case)
regex  PASS 51 ca (đề + edge case)
float() trả sai ở: ['inf', 'nan', 'Infinity']


In [20]:
import itertools
import random

alphabet = "1.eE+-a"
checked = 0
valid_count = 0
for n in range(1, 6):
    for tup in itertools.product(alphabet, repeat=n):
        t = "".join(tup)
        answers = {name: f(t) for name, f in CORRECT_METHODS.items()}
        answers["float_oracle"] = is_number_float(t)
        assert len(set(answers.values())) == 1, (t, answers)
        checked += 1
        valid_count += answers["flags"]
print(f"Vét cạn: {checked} chuỗi, {valid_count} chuỗi hợp lệ, 4 cách + float() trùng khớp hết")

random.seed(65)
pool = "0123456789" * 2 + "+-..eEx"
for _ in range(3000):
    t = "".join(random.choice(pool) for _ in range(random.randint(1, 20)))
    answers = {f(t) for f in CORRECT_METHODS.values()}
    assert len(answers) == 1, t
print("Ngẫu nhiên: 3000 chuỗi độ dài 1..20, 4 cách trùng khớp hết")

Vét cạn: 19607 chuỗi, 119 chuỗi hợp lệ, 4 cách + float() trùng khớp hết
Ngẫu nhiên: 3000 chuỗi độ dài 1..20, 4 cách trùng khớp hết


## 10. So sánh các cách

`n = len(s)`, theo đề `n <= 20`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Đúng đề? | Khi nào dùng |
|---|---|---|---|---|---|
| 1. `float()` + `try/except` | mượn ngữ pháp Python | `O(n)` | `O(n)` | **Sai** (`inf`, `nan`) | Chỉ để học bài học; làm "trọng tài" trên bảng chữ cái hạn chế |
| 2. Tách theo `e` | dịch thẳng ngữ pháp thành hàm | `O(n)` | `O(n)` | Đúng | Dễ nghĩ, dễ đọc, nộp được |
| 3. Một lượt + 3 cờ | nhớ 3 sự thật khi quét | `O(n)` | `O(1)` | Đúng | **Cách nên thuộc** cho phỏng vấn |
| 4. DFA | luật nằm trong bảng chuyển | `O(n)` | `O(1)` | Đúng | Khi luật phức tạp/hay đổi; muốn chứng minh chặt |
| 5. Regex `fullmatch` | mô tả mẫu bằng 1 dòng | thực tế `O(n)` | `O(1)` | Đúng | Code thực tế, kiểm tra input nhanh |

Thứ tự học hợp lý: **ngữ pháp → cách 2 → cách 3 → cách 4 → cách 5**: mỗi bước là một cách *mã hóa* cùng một ngữ pháp gọn hơn.

## 11. Lỗi người mới hay gặp

| # | Lỗi | Input làm lộ lỗi | Sửa |
|---|---|---|---|
| 1 | Quên `seen_digit = False` khi gặp `e` | `"1e"` → True (sai) | reset cờ khi gặp `e` |
| 2 | Chỉ cho dấu ở `i == 0`, quên vị trí sau `e` | `"3e+7"` → False (sai) | thêm điều kiện `s[i-1] in "eE"` |
| 3 | Cho dấu ở bất kỳ đâu | `"1+2"` → True (sai) | dấu chỉ ở đầu hoặc sau `e` |
| 4 | Không cấm dấu chấm sau `e` | `"99e2.5"` → True (sai) | `if seen_dot or seen_exp: return False` |
| 5 | Dùng `re.match` thay vì `fullmatch` | `"1a"` → khớp (sai) | dùng `re.fullmatch` |
| 6 | Dùng `re.match(p + "$")` | `"1\n"` → khớp (sai): `$` khớp cả trước ký tự xuống dòng cuối | dùng `fullmatch` hoặc `\Z` |
| 7 | Dùng `\d` / `isdigit()` | chữ số Unicode lọt qua | dùng `[0-9]` / `'0' <= c <= '9'` |
| 8 | Dùng `float()` | `"inf"`, `"nan"` → True (sai) | tự kiểm tra theo ngữ pháp |

Chạy thử vài bản sai để "thấy tận mắt":

In [21]:
def buggy_no_reset(s: str) -> bool:          # lỗi 1
    seen_digit = seen_dot = seen_exp = False
    for i, c in enumerate(s):
        if '0' <= c <= '9':
            seen_digit = True
        elif c in "+-":
            if i > 0 and s[i - 1] not in "eE":
                return False
        elif c == ".":
            if seen_dot or seen_exp:
                return False
            seen_dot = True
        elif c in "eE":
            if seen_exp or not seen_digit:
                return False
            seen_exp = True                  # QUÊN: seen_digit = False
        else:
            return False
    return seen_digit

print("Lỗi 1, '1e'  :", buggy_no_reset("1e"), "| đúng:", is_number_flags("1e"))
BODY = r"[+-]?(?:[0-9]+\.?[0-9]*|\.[0-9]+)(?:[eE][+-]?[0-9]+)?"
print("Lỗi 5, '1a'  : match ->", bool(re.match(BODY, "1a")), "| fullmatch ->", bool(re.fullmatch(BODY, "1a")))
print(r"Lỗi 6, '1\n' : match+$ ->", bool(re.match(BODY + "$", "1\n")), "| fullmatch ->", bool(re.fullmatch(BODY, "1\n")))

Lỗi 1, '1e'  : True | đúng: False
Lỗi 5, '1a'  : match -> True | fullmatch -> False
Lỗi 6, '1\n' : match+$ -> True | fullmatch -> False


## 12. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu nhận ra "bài kiểm tra định dạng chuỗi" (parsing / validation):**

- Đề cho một **định nghĩa nhiều tầng** ("X là Y theo sau bởi Z tùy chọn...") → viết lại thành ngữ pháp trước khi code.
- Câu hỏi là "chuỗi có hợp lệ không?" và mỗi ký tự chỉ cần xử lý một lần → nghĩ tới **một lượt quét + cờ** hoặc **DFA**.
- Có nhiều "giai đoạn" nối tiếp (dấu → phần nguyên → chấm → phần lẻ → mũ) → mỗi giai đoạn gần như là **một trạng thái**.
- Ngôn ngữ cho phép và không cấm → regex `fullmatch` là cách kiểm chứng nhanh nhất.

**Quy trình 4 bước khi gặp bài kiểu này:**

1. Viết ngữ pháp bằng ký hiệu gọn.
2. Liệt kê ví dụ đúng/sai **sát biên** cho từng luật (rỗng, chỉ có dấu, chấm không số, `e` ở đầu/cuối...).
3. Chọn cách mã hóa: cờ, DFA, hoặc regex.
4. Test vét cạn chuỗi ngắn trên bảng chữ cái nhỏ, đối chiếu ít nhất 2 cách.

**Bài liên quan:**

- LeetCode 8 — *String to Integer (atoi)*: cũng là quét trái-phải với dấu và chữ số, nhưng phải *đọc giá trị* và xử lý tràn số.
- LeetCode 468 — *Validate IP Address*: kiểm tra định dạng theo ngữ pháp, tách theo `.`/`:` giống cách 2.
- LeetCode 10 — *Regular Expression Matching* và 44 — *Wildcard Matching*: lần này **chính bạn** phải cài cỗ máy khớp mẫu (dùng DP).
- Biến thể đời thực: số trong **JSON** cấm `"4."`, `".5"`, `"+1"` và cấm số 0 đứng đầu như `"0089"`; chỉ cần sửa vài ô trong bảng DFA.

## 13. Bài tập vận dụng

Sáu bài từ dễ đến khó. Tự làm trước, gợi ý ở mục 14, đáp án ở **cuối notebook**.

### Bài 1 — Đoán output (không chạy code)

Với mỗi chuỗi, đoán `isNumber` trả gì, điền vào dict `predictions` (thay `None` bằng `True`/`False`):

`"0e"`, `".1."`, `"+.8"`, `"-e58"`, `"6e6.5"`, `"3.e+0"`, `"46.e3"`, `"+E3"`

In [22]:
predictions = {
    "0e": None,
    ".1.": None,
    "+.8": None,
    "-e58": None,
    "6e6.5": None,
    "3.e+0": None,
    "46.e3": None,
    "+E3": None,
}

# Điền xong rồi mới mở dòng dưới để tự chấm:
# for t, guess in predictions.items():
#     print(t, "đoán", guess, "| đúng", is_number_flags(t), "OK" if guess == is_number_flags(t) else "SAI")

### Bài 2 — Tính tay trên DFA

Không chạy code, lập bảng `Bước | Ký tự | Loại | Trạng thái trước | Trạng thái sau` cho chuỗi `"-.5E3"` theo ma trận chuyển ở mục 7.4. Kết thúc ở trạng thái nào? Chuỗi có hợp lệ không?

Làm tiếp với `"7.e"`: máy kẹt hay đọc hết? Kết luận?

### Bài 3 — Điền TODO cho cách 3

Hoàn thiện hàm một lượt quét với 3 cờ, không nhìn lại mục 6.

In [23]:
def exercise_is_number_flags(s: str) -> bool:
    seen_digit = seen_dot = seen_exp = False
    for i, c in enumerate(s):
        if '0' <= c <= '9':
            # TODO 1: cập nhật cờ
            pass
        elif c in "+-":
            # TODO 2: dấu chỉ hợp lệ ở đâu?
            pass
        elif c == ".":
            # TODO 3: khi nào dấu chấm bị cấm? cập nhật cờ
            pass
        elif c in "eE":
            # TODO 4: khi nào e bị cấm? cập nhật HAI cờ
            pass
        else:
            # TODO 5: ký tự khác
            pass
    # TODO 6: trả về gì khi hết chuỗi?
    raise NotImplementedError

# Gợi ý test (mở ra sau khi làm xong):
# for t, expected in ALL_CASES:
#     assert exercise_is_number_flags(t) == expected, (t, expected)

### Bài 4 — Biến thể: đề phiên bản cũ có khoảng trắng

Viết `is_number_with_spaces(s)`: cho phép **khoảng trắng `' '` ở đầu và cuối** (bao nhiêu cũng được), nhưng **không** cho khoảng trắng ở giữa. Ví dụ: `" 0.1 "` → True, `"  -3e2"` → True, `"1 2"` → False, `"   "` → False, `" . "` → False.

In [24]:
def is_number_with_spaces(s: str) -> bool:
    # TODO: bỏ khoảng trắng hai đầu rồi tái sử dụng một cách đã học
    raise NotImplementedError

# Gợi ý test:
# assert is_number_with_spaces(" 0.1 ") is True
# assert is_number_with_spaces("  -3e2") is True
# assert is_number_with_spaces("1 2") is False
# assert is_number_with_spaces("   ") is False
# assert is_number_with_spaces(" . ") is False

### Bài 5 — Biến thể: dấu gạch dưới phân cách chữ số (giống Python)

Mở rộng luật: cho phép ký tự `'_'` **chỉ khi nó nằm giữa hai chữ số** (ở bất kỳ nhóm digits nào, kể cả phần mũ).

- Hợp lệ: `"1_000"`, `"1_000.000_1"`, `"2e1_0"`, `".5_5"`
- Không hợp lệ: `"_1"`, `"1_"`, `"1__0"`, `"1_.5"`, `"1._5"`, `"1e_5"`, `"_"`

Viết `is_number_underscore(s)`.

In [25]:
def is_number_underscore(s: str) -> bool:
    # TODO: kiểm tra mỗi '_' có chữ số ở cả hai bên, rồi bỏ '_' và dùng lại hàm cũ
    raise NotImplementedError

# Gợi ý test:
# for t in ["1_000", "1_000.000_1", "2e1_0", ".5_5"]:
#     assert is_number_underscore(t) is True, t
# for t in ["_1", "1_", "1__0", "1_.5", "1._5", "1e_5", "_"]:
#     assert is_number_underscore(t) is False, t

### Bài 6 — Mini challenge: đếm chuỗi hợp lệ bằng DFA

Xét bảng chữ cái 15 ký tự: 10 chữ số, `+`, `-`, `.`, `e`, `E`. Có bao nhiêu chuỗi **độ dài đúng `n`** là số hợp lệ?

Với `n = 1` đáp án là `10` (chỉ các chữ số). Với `n = 2` đáp án là `140`. Hãy viết `count_valid_numbers(n)` chạy nhanh cả với `n = 20` (có `15^20 ≈ 3.3 × 10^23` chuỗi, không thể thử hết!).

In [26]:
def count_valid_numbers(n: int) -> int:
    # TODO: đếm theo TRẠNG THÁI của DFA thay vì theo từng chuỗi
    raise NotImplementedError

# Gợi ý test:
# assert count_valid_numbers(1) == 10
# assert count_valid_numbers(2) == 140

## 14. Gợi ý theo mức

**Bài 1.** (1) Áp 3 luật ở mục 2. (2) Sau `e` phải có integer; trước `e` phải có số. (3) Có thể chạy `trace_flags` sau khi đoán xong để kiểm tra.

**Bài 2.** (1) Đổi từng ký tự thành loại (`sign`, `dot`, `digit`, `exp`). (2) Tra hàng = trạng thái hiện tại, cột = loại. (3) `"-.5E3"` đi qua `sign`, `dot_no_int`, ... và kết thúc ở một trạng thái có chữ `int`.

**Bài 3.** (1) Nhìn bảng luật ở mục 6. (2) Dấu: `if i > 0 and s[i - 1] not in "eE": return False`. (3) Gặp `e`: cần `not seen_exp and seen_digit`, sau đó `seen_exp = True`, `seen_digit = False`; cuối hàm `return seen_digit`.

**Bài 4.** (1) Khoảng trắng ở giữa vẫn là "ký tự khác" nên cách cũ tự loại. (2) `s.strip(" ")` bỏ khoảng trắng hai đầu. (3) `return is_number_flags(s.strip(" "))`; chuỗi toàn khoảng trắng thành `""` và hàm cũ trả `False`.

**Bài 5.** (1) Hai bước: kiểm tra vị trí `'_'`, rồi xóa `'_'` và gọi hàm cũ. (2) Với mỗi `i` mà `s[i] == "_"`: cần `0 < i < len(s) - 1` và `s[i-1]`, `s[i+1]` đều là chữ số. (3) `"1__0"` bị loại vì `'_'` đầu tiên có hàng xóm phải là `'_'`.

**Bài 6.** (1) Hai chuỗi khác nhau nhưng đang ở **cùng trạng thái** thì "tương lai" giống hệt nhau, nên chỉ cần biết *có bao nhiêu chuỗi đang ở mỗi trạng thái*. (2) `dp = {"start": 1}`; mỗi bước, với mỗi `(state, ways)` và mỗi cạnh `kind -> nxt`, cộng `ways * (số ký tự thuộc kind)` vào `new[nxt]`. Số ký tự: digit 10, sign 2, dot 1, exp 2. (3) Đáp án = tổng `dp[state]` với `state` thuộc `ACCEPTING`; kiểm tra với vét cạn khi `n <= 3`.

## 15. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1:**

| `s` | Kết quả | Lý do |
|---|---|---|
| `"0e"` | False | sau `e` không có chữ số |
| `".1."` | False | hai dấu chấm |
| `"+.8"` | True | decimal dạng `. digits` có dấu |
| `"-e58"` | False | trước `e` không có chữ số |
| `"6e6.5"` | False | phần mũ có dấu chấm |
| `"3.e+0"` | True | decimal `"3."` + exponent `"e+0"` |
| `"46.e3"` | True | decimal `"46."` + exponent `"e3"` |
| `"+E3"` | False | trước `E` chỉ có dấu, chưa có số |

**Bài 2:** `"-.5E3"`: `start -sign-> sign -dot-> dot_no_int -digit-> frac -exp-> exp -digit-> exp_int`, kết thúc ở `exp_int` (chấp nhận) → **True**. `"7.e"`: `start -> int -> dot_after_int -> exp`, đọc hết nhưng `exp` không phải trạng thái chấp nhận → **False**. Cell dưới kiểm tra lại cả hai bài bằng code.

In [27]:
answer_predictions = {"0e": False, ".1.": False, "+.8": True, "-e58": False,
                      "6e6.5": False, "3.e+0": True, "46.e3": True, "+E3": False}
for t, expected in answer_predictions.items():
    assert is_number_flags(t) == expected == is_number_dfa(t), t
print("Bài 1 đáp án khớp code")

assert trace_dfa("-.5E3") is True
assert trace_dfa("7.e") is False

Bài 1 đáp án khớp code
s = '-.5E3'
  bước 1: '-' (sign )         start -> sign
  bước 2: '.' (dot  )          sign -> dot_no_int
  bước 3: '5' (digit)    dot_no_int -> frac
  bước 4: 'E' (exp  )          frac -> exp
  bước 5: '3' (digit)           exp -> exp_int
  hết chuỗi ở exp_int: chấp nhận -> True
s = '7.e'
  bước 1: '7' (digit)         start -> int
  bước 2: '.' (dot  )           int -> dot_after_int
  bước 3: 'e' (exp  ) dot_after_int -> exp
  hết chuỗi ở exp: không chấp nhận -> False


In [28]:
# Bài 3
def answer_is_number_flags(s: str) -> bool:
    seen_digit = seen_dot = seen_exp = False
    for i, c in enumerate(s):
        if '0' <= c <= '9':
            seen_digit = True
        elif c in "+-":
            if i > 0 and s[i - 1] not in "eE":
                return False
        elif c == ".":
            if seen_dot or seen_exp:
                return False
            seen_dot = True
        elif c in "eE":
            if seen_exp or not seen_digit:
                return False
            seen_exp, seen_digit = True, False
        else:
            return False
    return seen_digit

for t, expected in ALL_CASES:
    assert answer_is_number_flags(t) == expected, (t, expected)

# Bài 4
def answer_is_number_with_spaces(s: str) -> bool:
    return is_number_flags(s.strip(" "))

assert answer_is_number_with_spaces(" 0.1 ") is True
assert answer_is_number_with_spaces("  -3e2") is True
assert answer_is_number_with_spaces("1 2") is False
assert answer_is_number_with_spaces("   ") is False
assert answer_is_number_with_spaces(" . ") is False
print("Bài 3, Bài 4 PASS")

Bài 3, Bài 4 PASS


In [29]:
# Bài 5
def answer_is_number_underscore(s: str) -> bool:
    for i, c in enumerate(s):
        if c == "_":
            if not (0 < i < len(s) - 1):
                return False
            if not ('0' <= s[i - 1] <= '9' and '0' <= s[i + 1] <= '9'):
                return False
    return is_number_flags(s.replace("_", ""))

for t in ["1_000", "1_000.000_1", "2e1_0", ".5_5"]:
    assert answer_is_number_underscore(t) is True, t
for t in ["_1", "1_", "1__0", "1_.5", "1._5", "1e_5", "_"]:
    assert answer_is_number_underscore(t) is False, t

# Đối chiếu với float() của Python (cũng dùng luật '_' giữa hai chữ số),
# trên bảng chữ cái không có chữ cái nào ngoài e nên không vướng inf/nan.
random.seed(515)
for _ in range(3000):
    t = "".join(random.choice("11_.e+-") for _ in range(random.randint(1, 8)))
    assert answer_is_number_underscore(t) == is_number_float(t), t
print("Bài 5 PASS (kể cả 3000 chuỗi ngẫu nhiên đối chiếu với float())")

Bài 5 PASS (kể cả 3000 chuỗi ngẫu nhiên đối chiếu với float())


In [30]:
# Bài 6
KIND_COUNT = {"digit": 10, "sign": 2, "dot": 1, "exp": 2}

def answer_count_valid_numbers(n: int) -> int:
    dp = {"start": 1}                     # dp[state] = số chuỗi độ dài hiện tại đang ở state
    for _ in range(n):
        new = {}
        for state, ways in dp.items():
            for kind, nxt in TRANSITIONS[state].items():
                new[nxt] = new.get(nxt, 0) + ways * KIND_COUNT[kind]
        dp = new
    return sum(ways for state, ways in dp.items() if state in ACCEPTING)

ALPHABET15 = "0123456789+-.eE"
for n in range(1, 4):
    brute = sum(is_number_flags("".join(p)) for p in itertools.product(ALPHABET15, repeat=n))
    assert answer_count_valid_numbers(n) == brute, n
    print(f"n={n}: DP = {answer_count_valid_numbers(n)}, vét cạn = {brute}")
assert answer_count_valid_numbers(1) == 10 and answer_count_valid_numbers(2) == 140
print("n=20:", answer_count_valid_numbers(20))

n=1: DP = 10, vét cạn = 10
n=2: DP = 140, vét cạn = 140
n=3: DP = 1740, vét cạn = 1740
n=20: 1339200000000000000000


## 16. Template nên nhớ

Ngữ pháp trong đầu:

```text
number = [sign] (digits [. [digits]] | . digits) [ (e|E) [sign] digits ]
```

Bản nộp ngắn gọn (cách 3, một lượt + 3 cờ):

```python
class Solution:
    def isNumber(self, s: str) -> bool:
        seen_digit = seen_dot = seen_exp = False
        for i, c in enumerate(s):
            if '0' <= c <= '9':
                seen_digit = True
            elif c in "+-":
                if i > 0 and s[i - 1] not in "eE":
                    return False
            elif c == ".":
                if seen_dot or seen_exp:
                    return False
                seen_dot = True
            elif c in "eE":
                if seen_exp or not seen_digit:
                    return False
                seen_exp, seen_digit = True, False
            else:
                return False
        return seen_digit
```

Câu thần chú: **"Số thì bật cờ; dấu ở đầu hoặc sau e; chấm một lần và trước e; e một lần, cần số trước, reset số sau; cuối cùng phải có số."**

Bản regex (khi được phép):

```python
import re
NUMBER_RE = re.compile(r"[+-]?(?:[0-9]+\.?[0-9]*|\.[0-9]+)(?:[eE][+-]?[0-9]+)?")
return NUMBER_RE.fullmatch(s) is not None
```

## 17. Checklist trước khi submit

- [ ] Mình **không** dùng `float()`/`try-except` (sẽ sai với `"inf"`, `"nan"`).
- [ ] Chữ số kiểm tra bằng `'0' <= c <= '9'` hoặc `[0-9]`, không phải `isdigit()`/`\d`.
- [ ] Dấu `+`/`-` chỉ được ở `i == 0` hoặc ngay sau `e`/`E`.
- [ ] Dấu chấm: tối đa một lần **và** không được sau `e`.
- [ ] `e`/`E`: tối đa một lần, phải có chữ số trước, và **reset `seen_digit`** sau khi gặp.
- [ ] Hết chuỗi trả `seen_digit` (không phải `True`).
- [ ] Đã thử: `"."`, `"e"`, `"+"`, `"1e"`, `"e3"`, `".e1"`, `"4."`, `"-.9"`, `"3.e+0"`, `"99e2.5"`, `"--6"`, `"1+2"`, `"inf"`.
- [ ] Nếu dùng regex: `re.fullmatch`, không phải `re.match`/`re.search`.
- [ ] Bám đề hiện hành: không có khoảng trắng, không cần `strip()`.